# Express Links · Week 2, step 9: the evaluation code

**Project:** Express Links. Can adding a few well-placed links to the road graph help a traffic GNN (Graph WaveNet) warn about jams 30–60 minutes early?
**This notebook:** installs `metrics.py`, the one scoring script every run goes through, checks it by hand on a tiny example, checks it on the toy city with a perfect and a useless forecast, and scores whatever runs already exist in Drive.

**Owner: Mohammed (step 9).** Where this depends on other people's steps, the code says so.

---

## 1. Why this step matters

The proposal's claim is about **warnings**, not average error: does the model say "this road will jam" 30–60 minutes before it does? MAE alone cannot answer that, because a forecast can have a good MAE and still miss every jam. So every run, on every dataset, must be scored the same way: the same jam rule, the same warning rule, the same subsets. Writing that once, now, means the 84 runs of weeks 3–4 and the K sweep of week 5 all produce comparable rows.

---

## 2. The rules, in plain words

**A jam** (proposal 3.4): at a road, the speed falls by at least 20 km/h (**12.4 mph**) within 15 minutes and stays that low for 15 minutes. Zero readings are missing data, never a jam. A *jam start* is the first 5-minute step where this holds.

**A warning at H minutes** (H = 30 or 60): take the forecast made at time *a*. Stick the last 15 minutes of observed speeds in front of the 12 predicted steps, and apply the jam rule to that series. If it finds a jam *starting* at step H, the forecast warns of a jam at *a* + H. (At 60 minutes the forecast ends at that step, so "lasts 15 minutes" cannot be checked there; only the drop itself is.)

**Recall at H** = share of real jam starts that some forecast made about H minutes earlier (± 15 min) warned about.
**Precision at H** = share of warnings for which a real jam did start at that road about H minutes later (± 15 min).

**Standard errors**: MAE, RMSE, MAPE at 15, 30 and 60 minutes, and the 12-step average, masked exactly like Graph WaveNet (zeros ignored).

**Behind a bottleneck**, two cases:
- *Benchmarks (METR-LA, PEMS-BAY)*: a fixed set of roads, `behind_bottleneck` in step 5's `bottleneck/regions.npz`. Both recall and precision are reported for them.
- *Toy cities*: there is no fixed "behind" side, because jams start in both districts. A jam is behind the bottleneck when it **came from the other district**, which step 8's `*_jam_arrivals.csv` records (`crossed`). Recall is reported for those jams; precision is not defined for a per-jam subset and is left empty.

---

## 3. What this notebook produces

| File | What it is |
|---|---|
| `code/metrics.py` | the scoring module: `score_run(run_dir, data_dir)` → one results row, saved as `run_dir/score.json` |
| `runs/scores.csv` | one row per scored run (dataset, graph, seed, every metric); rerunning replaces a run's row |
| `runs/scores_summary.csv` | mean (std) over seeds per dataset × graph version |

---

## 4. Who depends on what

| Needs from others | Owner | Status |
|---|---|---|
| `data/processed/<dataset>/test.npz` | step 2 (Mohammed) | done |
| `data/processed/<benchmark>/bottleneck/regions.npz` | step 5 (Mohammed) | done |
| `data/raw/toy/<city>_jam_arrivals.csv` | step 8 (Mohammed) | done |
| `runs/<dataset>/<graph>/seed<k>/test_predictions.npz` | step 3 baseline (Mohammed, done); step 12 runs (together) | partly |
| the express graph for the week-2 checkpoint run | **step 7 (Al Zahraa)** | done |
| the seven graph versions to train and score | **step 10 (Al Zahraa)** | done |
| `data/raw/sharjah/SHARJAH_jam_arrivals.csv` | step 11 | done (read from `raw/sharjah`, not `raw/toy`) |

Nothing here waits on Al Zahraa's code: her graphs only change which run folders exist. The week-2 checkpoint (original graph vs express graph on `TOY-C1`, scored by this code) needs her step 7 output and one training run from step 12.

## 5. Connect Google Drive

Same folder as steps 2–8. Outside Colab it falls back to a local `express-links-local` folder (handy for a quick test).

In [1]:
import os, sys

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    CANDIDATES = [
        '/content/drive/MyDrive/AdvML_F26_Project_&_Survey_Paper/Project/Coding',
        '/content/drive/.shortcut-targets-by-id/1BXg48qqID3HutKK7oqQzm59_HW3HAVcM/AdvML_F26_Project_&_Survey_Paper/Project/Coding',
        '/content/drive/MyDrive/Project/Coding',
    ]
    PROJECT_DIR = next((p for p in CANDIDATES if os.path.isdir(p)), None)
    assert PROJECT_DIR, ("Couldn't find the 'Coding' folder. Mount Drive with b00108492@aus.edu, "
                         "or in Drive right-click 'AdvML_F26_Project_&_Survey_Paper' -> Organize -> Add shortcut -> My Drive, then rerun.")
else:
    PROJECT_DIR = os.path.abspath(os.environ.get('PROJECT_DIR', 'express-links-local'))
    os.makedirs(PROJECT_DIR, exist_ok=True)

import glob, json, time
import numpy as np
import pandas as pd

CODE_DIR = os.path.join(PROJECT_DIR, 'code')
DATA_ROOT = os.path.join(PROJECT_DIR, 'data', 'processed')
RAW_TOY = os.path.join(PROJECT_DIR, 'data', 'raw', 'toy')
RUNS_DIR = os.path.join(PROJECT_DIR, 'runs')
for d in (CODE_DIR, RUNS_DIR):
    os.makedirs(d, exist_ok=True)
print('PROJECT_DIR =', PROJECT_DIR)

Mounted at /content/drive
PROJECT_DIR = /content/drive/MyDrive/AdvML_F26_Project_&_Survey_Paper/Project/Coding


## 6. Install `metrics.py`

The whole module is in the next cell. Running it writes `code/metrics.py` to Drive, so a training notebook can do:

```python
import sys; sys.path.insert(0, os.path.join(PROJECT_DIR, 'code'))
import metrics
row = metrics.score_run(RUN_DIR, DATA_DIR)
```

If you change a rule, change it **here** and rerun this cell; never edit the copy in Drive by hand.

In [2]:
METRICS_SRC = r"""'''
Express Links · step 9 · metrics.py  (owner: Mohammed)

One scoring function that turns a run's forecasts into a results row:
  - masked MAE, RMSE, MAPE at 15, 30 and 60 minutes (all roads, and roads behind a bottleneck)
  - jam recall and precision at 30 and 60 minutes (all roads, and roads behind a bottleneck)

What it reads
  run_dir/test_predictions.npz   from a training run (step 3 / step 12): pred (S, 12, N) in mph, t_anchor (S,)
  data_dir/test.npz              from step 2 / step 8: x, y (S, 12, N, 2), t_index, t_anchor
  behind-a-bottleneck subset, one of:
    data_dir/bottleneck/regions.npz          step 5 (METR-LA, PEMS-BAY): per-road flag behind_bottleneck
    raw/toy/<dataset>_jam_arrivals.csv       step 8 (toy cities): per-jam flag 'crossed' (the jam came from the other district)
    raw/sharjah/<dataset>_jam_arrivals.csv   step 11 (Sharjah): the same file and flag

Depends on other people's steps only for inputs, not for code:
  step 7  (Al Zahraa)  express graph           -> needed for the week-2 checkpoint run, not for scoring
  step 10 (Al Zahraa)  seven graph versions    -> each version is just another run folder to score
  step 12 (together)   the 42 + 42 runs        -> score_many() loops over them

The jam rule (proposal 3.4)
  A jam starts at road n, time t when the speed at t is at least 12.4 mph (20 km/h) below the highest
  reading of the previous 15 minutes, and stays that low for the next 15 minutes. Zero readings are
  missing data: a jam is never declared on them.

The warning rule
  A forecast made at anchor a warns of a jam at road n, H minutes ahead, when the jam rule, applied to
  the series [last 15 min observed, then the 12 predicted steps], finds a jam START at step H: the
  predicted speed is at least 12.4 mph below the highest of the 3 steps before it, it was not already
  that low one step earlier, and it stays that low for the predicted steps that follow (3 steps at
  30 min; at 60 min the forecast ends there, so only the step itself is checked).
  Recall(H)    = share of real jam starts t for which some forecast made at a in [t-H-TOL, t-H+TOL] warned.
  Precision(H) = share of warnings (a, n) for which a real jam starts at road n in [a+H-TOL, a+H+TOL].
  TOL is 15 minutes (3 steps), the same granularity as the jam rule.
'''
import os, json
import numpy as np
import pandas as pd

STEP_MIN   = 5
JAM_DROP   = 12.4          # mph  (20 km/h)
WITHIN     = 3             # the drop happens within 15 min
LASTS      = 3             # and lasts 15 min
TOL        = 3             # timing tolerance for warnings, in steps (15 min)
HORIZONS   = {15: 2, 30: 5, 60: 11}       # minutes -> index in the 12-step forecast
WARN_H     = (30, 60)


# ---------------------------------------------------------------------------
# 1. Standard errors, masked like Graph WaveNet (null value 0 = missing reading)
# ---------------------------------------------------------------------------
def masked_errors(pred, true, null_val=0.0):
    '''MAE, RMSE, MAPE (in %) over entries where true != null_val.'''
    pred, true = np.asarray(pred, float), np.asarray(true, float)
    mask = true != null_val
    if not mask.any():
        return dict(mae=np.nan, rmse=np.nan, mape=np.nan)
    err = pred[mask] - true[mask]
    return dict(mae=float(np.abs(err).mean()),
                rmse=float(np.sqrt((err ** 2).mean())),
                mape=float((np.abs(err) / np.abs(true[mask])).mean() * 100))


# ---------------------------------------------------------------------------
# 2. The jam rule on an observed speed table (T, N)
# ---------------------------------------------------------------------------
def jam_state(speed):
    '''True at (t, n) when the jam rule holds there. speed: (T, N), zeros = missing.'''
    T, N = speed.shape
    state = np.zeros((T, N), bool)
    for t in range(WITHIN, T - LASTS):
        prev = speed[t - WITHIN:t]
        after = speed[t:t + LASTS + 1]
        ok = (prev > 0).all(axis=0) & (after > 0).all(axis=0)       # no missing readings involved
        level = prev.max(axis=0) - JAM_DROP
        state[t] = ok & (after <= level[None, :] + 1e-6).all(axis=0)
    return state


def jam_starts(speed):
    '''Jam starts: the first step of each run of jam_state. Returns a DataFrame with columns t, n.'''
    state = jam_state(speed)
    start = state & ~np.vstack([np.zeros((1, state.shape[1]), bool), state[:-1]])
    t, n = np.nonzero(start)
    return pd.DataFrame({'t': t, 'n': n})


# ---------------------------------------------------------------------------
# 3. Rebuild the test-period speed table from test.npz (no raw file needed)
# ---------------------------------------------------------------------------
def test_series(test_npz):
    '''Observed speeds (T_test, N) for the rows the test samples cover, plus the row offset.
    Row r of the result is raw row r + offset.'''
    x, y, t_index = test_npz['x'][..., 0], test_npz['y'][..., 0], test_npz['t_index']
    seq_in, seq_out = x.shape[1], y.shape[1]
    first, last = int(t_index[0]) - (seq_in - 1), int(t_index[-1]) + seq_out
    speed = np.full((last - first + 1, x.shape[2]), np.nan, np.float32)
    speed[0:seq_in] = x[0]
    for s in range(len(t_index)):
        r = int(t_index[s]) - first
        speed[r + 1:r + 1 + seq_out] = y[s]
    assert not np.isnan(speed).any(), 'test samples are not consecutive'
    return speed, first


# ---------------------------------------------------------------------------
# 4. Warnings and their scoring
# ---------------------------------------------------------------------------
def warnings_at(pred, x_obs, h_index):
    '''(S, N) bool: the forecast made at anchor s predicts a jam STARTING h_index+1 steps ahead.
    The jam rule is applied to the series [last 15 min observed, then the 12 predicted steps]:
    the predicted speed at that step is >= 12.4 mph below the highest of the 3 steps before it,
    it was not already that low one step earlier, and it stays that low for the next 3 predicted
    steps (fewer at the end of the forecast: at 60 min only the step itself can be checked).
    pred: (S, 12, N) mph; x_obs: (S, 12, N) observed inputs (last row = anchor).'''
    series = np.concatenate([x_obs[:, -WITHIN:, :], pred], axis=1)       # (S, 3 + 12, N)
    k = WITHIN + h_index
    prev = series[:, k - WITHIN:k, :]
    after = series[:, k:min(k + LASTS + 1, series.shape[1]), :]
    ok = (prev > 0).all(axis=1)
    level = prev.max(axis=1) - JAM_DROP
    not_jammed_yet = prev[:, -1, :] > level + 1e-6
    return ok & not_jammed_yet & (after <= level[:, None, :] + 1e-6).all(axis=1)


def recall_precision(jams, warn, t_index, h_steps, road_mask=None, jam_mask=None):
    '''jams: DataFrame (t, n) of real jam starts in raw-row units; warn: (S, N) warnings whose anchors
    are raw rows t_index. road_mask (N,) or jam_mask (len(jams),) restrict the subset.'''
    S, N = warn.shape
    anchor_row = {int(r): s for s, r in enumerate(t_index)}
    rows = np.asarray(t_index)

    # a lookup of jam starts per road, for precision
    starts_by_road = {n: np.sort(g['t'].values) for n, g in jams.groupby('n')}

    # recall: for each real jam, any warning anchored in [t-h-TOL, t-h+TOL]?
    keep = np.ones(len(jams), bool)
    if road_mask is not None:
        keep &= road_mask[jams['n'].values]
    if jam_mask is not None:
        keep &= jam_mask
    hit = []
    for t, n in jams.loc[keep, ['t', 'n']].values:
        found = False
        for a in range(t - h_steps - TOL, t - h_steps + TOL + 1):
            s = anchor_row.get(a)
            if s is not None and warn[s, n]:
                found = True
                break
        hit.append(found)
    n_jams = int(keep.sum())
    recall = float(np.mean(hit)) if n_jams else np.nan

    # precision: for each warning, a real jam start at that road in [a+h-TOL, a+h+TOL]?
    # Only defined for a fixed set of roads. For a per-jam subset (toy cities: "the jam came from the
    # other district") a false warning belongs to no jam, so precision is reported as NaN there.
    if jam_mask is not None:
        n_warn = 0
        return dict(recall=recall, precision=np.nan, jams=n_jams, warnings=n_warn,
                    hits=int(np.sum(hit)) if n_jams else 0, correct=0)
    ws, wn = np.nonzero(warn)
    if road_mask is not None:
        sel = road_mask[wn]
        ws, wn = ws[sel], wn[sel]
    correct = []
    for s, n in zip(ws, wn):
        target = rows[s] + h_steps
        st = starts_by_road.get(n)
        correct.append(st is not None and np.any((st >= target - TOL) & (st <= target + TOL)))
    n_warn = len(correct)
    precision = float(np.mean(correct)) if n_warn else np.nan
    return dict(recall=recall, precision=precision, jams=n_jams, warnings=n_warn,
                hits=int(np.sum(hit)) if n_jams else 0, correct=int(np.sum(correct)) if n_warn else 0)


# ---------------------------------------------------------------------------
# 5. The behind-a-bottleneck subset
# ---------------------------------------------------------------------------
def load_subset(data_dir, jams=None, row_offset=0, raw_toy_dir=None):
    '''Returns (road_mask or None, jam_mask or None, source description).
    Benchmarks: step 5 regions.npz gives a per-road flag.
    Toy cities: step 8 jam_arrivals.csv says, per logged jam, whether it crossed the bottleneck.'''
    reg = os.path.join(data_dir, 'bottleneck', 'regions.npz')
    if os.path.exists(reg):
        r = np.load(reg, allow_pickle=True)
        return r['behind_bottleneck'].astype(bool), None, 'step 5 regions.npz (behind_bottleneck)'

    name = os.path.basename(os.path.normpath(data_dir))
    raw_dir = os.path.join(os.path.dirname(os.path.dirname(data_dir)), 'raw')
    # toy cities live in raw/toy, Sharjah in raw/sharjah (step 11): look in both
    folders = [raw_toy_dir] if raw_toy_dir else []
    folders += [os.path.join(raw_dir, 'toy'), os.path.join(raw_dir, 'sharjah')]
    arr_path = next((os.path.join(d, f'{name}_jam_arrivals.csv') for d in folders
                     if os.path.exists(os.path.join(d, f'{name}_jam_arrivals.csv'))), '')
    if arr_path and jams is not None:
        arr = pd.read_csv(arr_path)
        # match each detected jam start to the nearest logged arrival at the same road (within 2*TOL)
        crossed = np.zeros(len(jams), bool)
        matched = np.zeros(len(jams), bool)
        by_road = {n: g.sort_values('arrival_row') for n, g in arr.groupby('road')}
        for i, (t, n) in enumerate(jams[['t', 'n']].values):
            g = by_road.get(n)
            if g is None:
                continue
            d = np.abs(g['arrival_row'].values - (t + row_offset))
            k = int(np.argmin(d))
            if d[k] <= 2 * TOL:
                matched[i] = True
                crossed[i] = bool(g['crossed'].values[k])
        return None, crossed, f'step 8 {name}_jam_arrivals.csv (crossed; {matched.mean():.0%} of detected jams matched to the log)'
    return None, None, 'none found: behind-bottleneck columns will be empty'


# ---------------------------------------------------------------------------
# 6. Score one run
# ---------------------------------------------------------------------------
def score_run(run_dir, data_dir, raw_toy_dir=None, save=True):
    '''Returns one results row (dict) and writes run_dir/score.json.'''
    P = np.load(os.path.join(run_dir, 'test_predictions.npz'))
    D = np.load(os.path.join(data_dir, 'test.npz'))
    pred = P['pred'].astype(np.float32)                    # (S, 12, N) mph
    x_obs, y_true, t_index = D['x'][..., 0], D['y'][..., 0], D['t_index'].astype(int)
    assert pred.shape == y_true.shape, f'pred {pred.shape} vs test y {y_true.shape}'
    assert np.array_equal(P['t_anchor'], D['t_anchor']), 'run and test set have different anchors'

    speed, offset = test_series(D)
    jams = jam_starts(speed)
    jams['t'] = jams['t'] + offset                          # raw-row units, like t_index
    road_mask, jam_mask, subset_src = load_subset(data_dir, jams, 0, raw_toy_dir)

    row = {}
    cfg = os.path.join(run_dir, 'results.json')
    if os.path.exists(cfg):
        with open(cfg) as f:
            c = json.load(f)
        row.update({k: c.get(k) for k in ('dataset', 'graph', 'seed')})
    row.update(run=run_dir, subset_source=subset_src, test_samples=int(len(t_index)), roads=int(pred.shape[2]))

    # standard errors
    for mins, h in HORIZONS.items():
        e = masked_errors(pred[:, h], y_true[:, h])
        row.update({f'{k}_{mins}': v for k, v in e.items()})
        if road_mask is not None:
            e = masked_errors(pred[:, h][:, road_mask], y_true[:, h][:, road_mask])
            row.update({f'{k}_{mins}_behind': v for k, v in e.items()})
    e = masked_errors(pred, y_true)
    row.update({f'{k}_avg': v for k, v in e.items()})

    # jam warnings
    for mins in WARN_H:
        h = HORIZONS[mins]
        warn = warnings_at(pred, x_obs, h)
        r = recall_precision(jams, warn, t_index, h + 1)
        row.update({f'jam_recall_{mins}': r['recall'], f'jam_precision_{mins}': r['precision'],
                    f'jams_{mins}': r['jams'], f'warnings_{mins}': r['warnings']})
        if road_mask is not None or jam_mask is not None:
            r = recall_precision(jams, warn, t_index, h + 1, road_mask, jam_mask)
            row.update({f'jam_recall_{mins}_behind': r['recall'], f'jam_precision_{mins}_behind': r['precision'],
                        f'jams_{mins}_behind': r['jams'], f'warnings_{mins}_behind': r['warnings']})
    row['real_jams_in_test'] = int(len(jams))

    if save:
        with open(os.path.join(run_dir, 'score.json'), 'w') as f:
            json.dump(row, f, indent=2)
    return row


def score_many(run_dirs, data_root, raw_toy_dir=None, table_path=None):
    '''Score several run folders (each must have results.json naming its dataset) into one table.'''
    rows = []
    for rd in run_dirs:
        with open(os.path.join(rd, 'results.json')) as f:
            ds = json.load(f)['dataset']
        rows.append(score_run(rd, os.path.join(data_root, ds), raw_toy_dir))
    table = pd.DataFrame(rows)
    if table_path:
        if os.path.exists(table_path):
            old = pd.read_csv(table_path)
            old = old[~old['run'].isin(table['run'])]
            table = pd.concat([old, table], ignore_index=True)
        table.to_csv(table_path, index=False)
    return table


def summarise(table, by=('dataset', 'graph')):
    '''Mean (std) over seeds for the headline columns.'''
    cols = [c for c in table.columns if c.startswith(('mae_', 'rmse_', 'mape_', 'jam_'))]
    g = table.groupby(list(by))[cols]
    m, s = g.mean(), g.std()
    out = m.copy().astype(object)
    for c in cols:
        out[c] = [f'{a:.3f} ({b:.3f})' if not np.isnan(b) else f'{a:.3f}' for a, b in zip(m[c], s[c])]
    out['seeds'] = g.size()
    return out
"""

with open(os.path.join(CODE_DIR, 'metrics.py'), 'w') as f:
    f.write(METRICS_SRC)
sys.path.insert(0, CODE_DIR)
import importlib, metrics
importlib.reload(metrics)
M = metrics
print('metrics.py written to', os.path.join(CODE_DIR, 'metrics.py'))
print('jam rule: drop >=', M.JAM_DROP, 'mph within', M.WITHIN * 5, 'min, lasting', M.LASTS * 5, 'min; warning tolerance ±', M.TOL * 5, 'min')

metrics.py written to /content/drive/MyDrive/AdvML_F26_Project_&_Survey_Paper/Project/Coding/code/metrics.py
jam rule: drop >= 12.4 mph within 15 min, lasting 15 min; warning tolerance ± 15 min


## 7. Check by hand on a small example

Three roads, five hours, one real jam. Every number below was worked out by hand before the code ran: one jam (road 0 at step 30), two warnings at 30 minutes of which one is right (precision 0.5), one warning at 60 minutes that is right, MAE exactly 1 when the forecast is the truth plus 1, and a zero reading that must be ignored.

In [3]:

T, N = 60, 3
speed = np.full((T, N), 60.0, np.float32)
speed[30:41, 0] = 40.0        # road 0: a 20 mph drop at t=30 that lasts 55 min  -> a jam
speed[30:41, 1] = 50.0        # road 1: only a 10 mph drop                           -> not a jam
speed[45:47, 2] = 40.0        # road 2: a 20 mph drop that lasts only 10 min         -> not a jam
speed[10, 2] = 0.0            # a missing reading, must be ignored

# --- the jam rule ---
jams = M.jam_starts(speed)
assert jams.to_dict('records') == [{'t': 30, 'n': 0}], jams
print('OK  jam rule finds exactly one jam, at road 0, t=30')

# --- a test set in the step-2 layout: anchors 11..47 ---
anchors = np.arange(11, T - 12)
x = np.stack([speed[a - 11:a + 1] for a in anchors])     # (S, 12, N)
y = np.stack([speed[a + 1:a + 13] for a in anchors])
test = dict(x=x[..., None], y=y[..., None], t_index=anchors, t_anchor=anchors)
rebuilt, off = M.test_series(test)
assert off == 0 and np.array_equal(rebuilt, speed)
print('OK  the test-period speed table is rebuilt exactly from x and y')

# --- forecasts: persistence everywhere, then a few planted warnings ---
pred = np.repeat(x[:, -1:, :], 12, axis=1).copy()        # predict "no change"
s = {a: i for i, a in enumerate(anchors)}
pred[s[24], 5:9, 0] = 40.0     # made 30 min before the jam: predicts the drop at step 6, lasting 15 min -> true warning at 30
pred[s[17], 11, 0] = 40.0      # made 65 min before (within the 15-min tolerance)    -> true warning at 60
pred[s[20], 5:9, 2] = 40.0     # road 2 never jams                                   -> false warning at 30

jams['t'] = jams['t'] + off
for mins, exp in [(30, dict(recall=1.0, precision=0.5, jams=1, warnings=2)),
                  (60, dict(recall=1.0, precision=1.0, jams=1, warnings=1))]:
    h = M.HORIZONS[mins]
    warn = M.warnings_at(pred, x, h)
    r = M.recall_precision(jams, warn, anchors, h + 1)
    got = {k: r[k] for k in exp}
    assert got == exp, (mins, got, exp)
    print(f'OK  {mins} min: recall {r["recall"]:.2f}, precision {r["precision"]:.2f} ({r["warnings"]} warnings, {r["jams"]} jam)')

# behind-bottleneck subset = road 0 only: the false warning on road 2 drops out
road_mask = np.array([True, False, False])
r = M.recall_precision(jams, M.warnings_at(pred, x, 5), anchors, 6, road_mask=road_mask)
assert (r['recall'], r['precision'], r['warnings']) == (1.0, 1.0, 1)
print('OK  behind-bottleneck subset (road 0): recall 1.00, precision 1.00')

# --- errors ---
e = M.masked_errors(y + 1.0, y)
assert abs(e['mae'] - 1) < 1e-6 and abs(e['rmse'] - 1) < 1e-6
print(f'OK  pred = truth + 1 gives MAE 1.000, RMSE 1.000, MAPE {e["mape"]:.3f}%')
e = M.masked_errors(np.array([5.0, 7.0]), np.array([0.0, 10.0]))
assert e['mae'] == 3 and e['rmse'] == 3 and abs(e['mape'] - 30) < 1e-9
print('OK  the zero reading is masked: MAE 3, MAPE 30% from the one real reading')
print('ALL HAND CHECKS PASSED')


OK  jam rule finds exactly one jam, at road 0, t=30
OK  the test-period speed table is rebuilt exactly from x and y
OK  30 min: recall 1.00, precision 0.50 (2 warnings, 1 jam)
OK  60 min: recall 1.00, precision 1.00 (1 warnings, 1 jam)
OK  behind-bottleneck subset (road 0): recall 1.00, precision 1.00
OK  pred = truth + 1 gives MAE 1.000, RMSE 1.000, MAPE 1.797%
OK  the zero reading is masked: MAE 3, MAPE 30% from the one real reading
ALL HAND CHECKS PASSED


## 8. Check on the toy city: a perfect forecast and a useless one

Two fake "runs" on `TOY-C1` (from step 8):

- **oracle**: the forecast *is* the true future. Recall and precision should be 1.0 (a little lower at 60 minutes, where the end of the forecast cuts the "lasts 15 minutes" check).
- **persistence**: the forecast is "nothing changes". It can never predict a drop, so recall must be 0 and there are no warnings.

Any real model should land between these two. The cell is skipped if step 8's files are not in Drive yet.

In [4]:
toy_dir = os.path.join(DATA_ROOT, 'TOY-C1')
if not os.path.exists(os.path.join(toy_dir, 'test.npz')):
    print('TOY-C1 not found: run step 8 first, then rerun this cell')
else:
    D = np.load(os.path.join(toy_dir, 'test.npz'))
    x, y = D['x'][..., 0], D['y'][..., 0]
    fakes = {'oracle': y, 'persistence': np.repeat(x[:, -1:, :], 12, axis=1)}
    rows = []
    for name, pred in fakes.items():
        rd = os.path.join(RUNS_DIR, 'TOY-C1', f'check-{name}', 'seed0')
        os.makedirs(rd, exist_ok=True)
        np.savez_compressed(os.path.join(rd, 'test_predictions.npz'), pred=pred.astype(np.float32),
                            t_anchor=D['t_anchor'], horizon_min=np.arange(5, 61, 5))
        with open(os.path.join(rd, 'results.json'), 'w') as f:
            json.dump(dict(dataset='TOY-C1', graph=f'check-{name}', seed=0), f)
        t0 = time.time()
        rows.append(M.score_run(rd, toy_dir, RAW_TOY))
        print(f'{name}: scored in {time.time() - t0:.0f} s')
    show = ['graph', 'mae_60', 'jam_recall_30', 'jam_precision_30', 'jam_recall_60', 'jam_precision_60',
            'jam_recall_30_behind', 'jam_recall_60_behind', 'jams_30', 'jams_30_behind', 'real_jams_in_test']
    chk = pd.DataFrame(rows)[show].set_index('graph')
    display(chk.round(3))
    o, p = chk.loc['check-oracle'], chk.loc['check-persistence']
    assert o['jam_recall_30'] > 0.95 and o['jam_precision_30'] > 0.95, 'the oracle should get almost every jam'
    assert o['jam_recall_60'] > 0.9 and o['jam_precision_60'] > 0.8
    assert p['jam_recall_30'] == 0 and p['jam_recall_60'] == 0, 'persistence can never warn'
    assert o['mae_60'] == 0
    print('OK  oracle and persistence score as expected')
    print(rows[0]['subset_source'])

oracle: scored in 4 s
persistence: scored in 3 s


,mae_60,jam_recall_30,jam_precision_30,jam_recall_60,jam_precision_60,jam_recall_30_behind,jam_recall_60_behind,jams_30,jams_30_behind,real_jams_in_test
graph,,,,,,,,,,
check-oracle,0.000,1.0,1.0,1.0,0.897,1.0,1.0,3095,87,3095
check-persistence,4.404,0.0,NaN,0.0,NaN,0.0,0.0,3095,87,3095


OK  oracle and persistence score as expected
step 8 TOY-C1_jam_arrivals.csv (crossed; 100% of detected jams matched to the log)


## 9. Score every run already in Drive

Finds every `runs/<dataset>/<graph>/seed<k>/test_predictions.npz` (the step-3 baseline, and later all step-12 runs), scores it, and writes `runs/scores.csv` and `runs/scores_summary.csv`. Rerun this cell whenever new runs finish; a run's row is replaced, not duplicated. The two `check-*` fakes from section 8 are left out of the tables.

In [6]:
run_dirs = sorted(os.path.dirname(p) for p in glob.glob(os.path.join(RUNS_DIR, '*', '*', 'seed*', 'test_predictions.npz'))
                  if '/check-' not in p.replace(os.sep, '/'))
print(f'{len(run_dirs)} run folder(s) with forecasts:')
for rd in run_dirs:
    print('  ', os.path.relpath(rd, RUNS_DIR))

if run_dirs:
    table = M.score_many(run_dirs, DATA_ROOT, RAW_TOY, table_path=os.path.join(RUNS_DIR, 'scores.csv'))
    headline = ['dataset', 'graph', 'seed', 'mae_15', 'mae_30', 'mae_60', 'rmse_60', 'mape_60',
                'jam_recall_30', 'jam_precision_30', 'jam_recall_60', 'jam_precision_60',
                'jam_recall_60_behind', 'jam_precision_60_behind', 'mae_60_behind', 'real_jams_in_test']
    display(table[[c for c in headline if c in table.columns]].round(3))
    summary = M.summarise(table)
    summary.to_csv(os.path.join(RUNS_DIR, 'scores_summary.csv'))
    display(summary)
else:
    print('No runs yet. The step-3 baseline writes runs/METR-LA/original/seed0/test_predictions.npz; step 12 adds the rest.')

2 run folder(s) with forecasts:
   METR-LA/original/seed0
   TOY-C1/original/seed0


,dataset,graph,seed,mae_15,mae_30,mae_60,rmse_60,mape_60,jam_recall_30,jam_precision_30,jam_recall_60,jam_precision_60,jam_recall_60_behind,jam_precision_60_behind,mae_60_behind,real_jams_in_test
0,METR-LA,original,0,2.707,3.096,3.579,7.41,10.180,0.198,0.579,0.087,0.504,0.0,NaN,2.471,6363
1,TOY-C1,original,0,0.889,1.108,1.539,3.44,4.097,0.555,0.875,0.017,0.800,0.0,NaN,NaN,3095


,,mae_15,rmse_15,mape_15,mae_15_behind,rmse_15_behind,mape_15_behind,mae_30,rmse_30,mape_30,mae_30_behind,...,mape_avg,jam_recall_30,jam_precision_30,jam_recall_30_behind,jam_precision_30_behind,jam_recall_60,jam_precision_60,jam_recall_60_behind,jam_precision_60_behind,seeds
dataset,graph,,,,,,,,,,,,,,,,,,,,,
METR-LA,original,2.707,5.186,6.983,2.364,3.663,4.342,3.096,6.250,8.475,2.402,...,8.336,0.198,0.579,0.000,nan,0.087,0.504,0.000,nan,1
TOY-C1,original,0.889,1.314,1.755,nan,nan,nan,1.108,2.066,2.471,nan,...,2.631,0.555,0.875,0.621,nan,0.017,0.800,0.000,nan,1


## 10. How the results row is laid out

Every column that `score_run` returns, so the tables in weeks 3–5 need no new code:

| Column | Meaning |
|---|---|
| `dataset`, `graph`, `seed` | read from the run's `results.json` (step 3 writes it) |
| `mae_15`, `mae_30`, `mae_60`, `rmse_*`, `mape_*`, `*_avg` | masked errors, all roads |
| `*_behind` versions of the above | the same on roads behind a bottleneck (benchmarks only) |
| `jam_recall_30`, `jam_precision_30`, `jam_recall_60`, `jam_precision_60` | warnings, all roads |
| `jam_recall_*_behind`, `jam_precision_*_behind` | warnings for jams behind a bottleneck (precision empty for toy cities) |
| `jams_*`, `warnings_*` | the counts behind each rate, so a 100% on 3 jams is visible as such |
| `real_jams_in_test` | how many jam starts the rule found in the test period |
| `subset_source` | which file defined "behind a bottleneck" |

**Next steps that use this**

- Step 12 (together): each training notebook calls `metrics.score_run` at the end, exactly as section 6 shows, so a run is scored the moment it finishes.
- Step 16 (Mohammed): `scores.csv` → the main results table (mean ± std over seeds) and the jam-recall-at-60-min figure.
- Step 17 (Mohammed): the paired bootstrap, express vs same-side, runs on the per-jam hits; `recall_precision` returns the hit counts it needs.
- Week-2 checkpoint: needs **Al Zahraa's step 7** express graph and one `TOY-C1` run on it, then section 9 scores it next to the original-graph run.